# NEXUS — Colab GPU + Ollama + Google Drive

Notebook único para os **Dias 1–5** do `overcyber/minicurso-mult-agents`. O Dia 1 já havia sido testado; Dias 2–5 devem ser validados ponta a ponta aqui. Persistência: código/modelos/cache no Drive; Chroma/SQLite executam em `/content` e são sincronizados para o Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,shutil,subprocess,time,json,re,uuid,importlib,requests
D=Path('/content/drive/MyDrive/minicurso-mult-agents-colab'); DR=D/'repo'; DS=D/'state'; OM=D/'ollama/models'; HF=D/'huggingface'
R=Path('/content/minicurso-mult-agents'); N=R/'codigo/nexus'
for p in (D,DS,OM,HF): p.mkdir(parents=True,exist_ok=True)
def run(c,check=True,capture=False,env=None,cwd=None): return subprocess.run(c,shell=isinstance(c,str),check=check,text=True,capture_output=capture,env=env,cwd=cwd)
g=run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],False,True)
if g.returncode: raise RuntimeError('Selecione uma GPU em Runtime > Change runtime type > GPU')
print(g.stdout)

## Setup persistente, dependências e Ollama

In [ ]:
URL='https://github.com/overcyber/minicurso-mult-agents.git'
if not (DR/'.git').exists(): run(['git','clone',URL,str(DR)])
elif not run(['git','-C',str(DR),'status','--porcelain'],capture=True).stdout.strip():
    run(['git','-C',str(DR),'pull','--ff-only','origin','main'])
if R.exists(): shutil.rmtree(R)
shutil.copytree(DR,R,ignore=shutil.ignore_patterns('.git','__pycache__','*.pyc','.chroma','.chroma_hf','*.db','saida','tracos'))
os.chdir(N)
cons=Path('/content/nexus-constraints.txt'); cons.write_text('langchain==1.4.0\nlanggraph==1.2.11\nlanggraph-checkpoint-sqlite==3.1.1\nlangchain-ollama==1.1.0\nlangchain-chroma==1.1.0\nchromadb==1.5.9\n')
run([sys.executable,'-m','pip','install','-q','-r',str(N/'requirements.txt'),'-c',str(cons),'accelerate>=1.0'])
os.environ.update(HF_HOME=str(HF),HF_HUB_CACHE=str(HF/'hub'),TRANSFORMERS_CACHE=str(HF/'transformers'),SENTENCE_TRANSFORMERS_HOME=str(HF/'sentence-transformers'),OLLAMA_MODELS=str(OM),OLLAMA_HOST='127.0.0.1:11434')
if not shutil.which('ollama'): run('curl -fsSL https://ollama.com/install.sh | sh')
def alive():
    try: return requests.get('http://127.0.0.1:11434/api/tags',timeout=2).ok
    except: return False
if not alive():
    log=open(N/'ollama.log','ab',buffering=0); proc=subprocess.Popen(['ollama','serve'],env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
    for _ in range(60):
        if alive(): break
        time.sleep(1)
    else: raise RuntimeError('Ollama não iniciou; veja ollama.log')
def pull(m):
    names={x['name'] for x in requests.get('http://127.0.0.1:11434/api/tags').json().get('models',[])}
    if m not in names: run(['ollama','pull',m],env=os.environ.copy())
for m in ('qwen3:4b','nomic-embed-text'): pull(m)
os.environ.update(MODELO='qwen3:4b',MODELO_PEQUENO='qwen3:4b',MODELO_MEDIO='qwen3:4b',MODELO_GRANDE='qwen3:4b')
print(run(['ollama','run','qwen3:4b','Responda apenas OK.'],capture=True,env=os.environ.copy()).stdout)
print(run(['ollama','ps'],False,True,env=os.environ.copy()).stdout)

In [ ]:
STATE_DIRS=['.chroma','.chroma_hf','.chroma_hf_gpu','saida','tracos']
def cp(a,b):
    if not a.exists(): return
    if a.is_dir():
        if b.exists(): shutil.rmtree(b)
        shutil.copytree(a,b)
    else: b.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(a,b)
def restore():
    for x in STATE_DIRS: cp(DS/x,N/x)
    for pat in ('*.db','*.db-wal','*.db-shm','*.csv','*.json','*.jsonl','ollama.log'):
        for a in DS.glob(pat): cp(a,N/a.name)
def persist():
    for x in STATE_DIRS: cp(N/x,DS/x)
    for pat in ('*.db','*.db-wal','*.db-shm','*.csv','*.json','*.jsonl','ollama.log'):
        for a in N.glob(pat): cp(a,DS/a.name)
restore()
MODS={'agente','clientes','ferramentas','indexar','nexus','hooks','steering','ferramentas_web','equipe','prompts','handoff','memoria_semantica','email_assistente','app_gradio','avaliacao','modelos','roteador','pipelines','embeddings_hf','cli'}
def dia(n):
    for m in list(sys.modules):
        if m in MODS: sys.modules.pop(m,None)
    ps=[str(N/f'dia{i}') for i in range(1,6)]; sys.path[:]=[p for p in sys.path if p not in ps]; sys.path.insert(0,str(N/n)); importlib.invalidate_caches(); os.chdir(N)
run([sys.executable,'-m','compileall','-q',str(N)])
t=run([sys.executable,'-m','pytest','-q',str(N/'testes')],False,True,cwd=str(N)); print(t.stdout); print(t.stderr)

# Dia 1 — Python puro + tools
Arquivos: `agente.py`, `clientes.py`, `ferramentas.py`. O path confinement atual usa `Path.relative_to()`.

In [ ]:
dia('dia1'); import ferramentas as f1
print(f1.calcular('950 * 24')); print(f1.ler_arquivo('../../etc/passwd'))
import agente as a1
print(a1.rodar('Qual foi o faturamento de 2024? Leia o documento necessário e cite o nome dele.',backend='ollama',verbose=True)); persist()

# Dia 2 — LangChain + RAG/Chroma
Atenção: `ferramentas.py` regride para validação por `startswith`; `.chroma` não tem fingerprint do corpus/configuração.

In [ ]:
dia('dia2'); import indexar as i2
b=i2.construir(recriar=not (N/'.chroma').exists())
for d in b.similarity_search('faturamento de 2024',k=2): print(Path(d.metadata.get('source','?')).name,d.page_content[:400])
import agente as a2; print(a2.rodar('Qual foi o faturamento de 2024? Cite a fonte.')); persist()

# Dia 3 — LangGraph + HITL + hooks/steering
**Lacuna atual:** `hooks.py` e `steering.py` são testados isoladamente, mas `nexus.py` usa `ToolNode` diretamente e não os conecta ao grafo principal. Ferramentas web também precisam de proteção SSRF/prompt-injection em produção.

In [ ]:
dia('dia3'); import hooks,steering
print(hooks.avaliar_politicas('ler_arquivo',{'caminho':'../../etc/passwd'},{})); print(steering.detectar_estagnacao({'passos':7,'achados':[]}))
from langchain_core.messages import HumanMessage
import nexus as n3
g3=n3.compilar(str(N/'nexus_colab.db'),aprovar_ferramentas=False)
r=g3.invoke({'messages':[HumanMessage('Qual foi o faturamento de 2024? Cite a fonte.')],'passos':0},{'configurable':{'thread_id':'colab-d3'},'recursion_limit':30})
print(r['messages'][-1].content); persist()

# Dia 4 — Equipe multiagente
**Lacunas atuais:** `handoff.py` não participa da equipe principal; memória semântica é `InMemoryStore`; `lembrar()` não persiste; `email_assistente.processar()` não injeta retriever por padrão.

In [ ]:
dia('dia4'); from langchain_core.messages import HumanMessage; import equipe as e4
g4=e4.compilar(str(N/'equipe_colab.db')); q='Compare os fornecedores e recomende um, citando fontes.'
e={'messages':[HumanMessage(q)],'pergunta':q,'proximo':'','instrucao':q,'achados':[],'rascunho':'','veredito':'','rodadas':0}
r4=g4.invoke(e,{'configurable':{'thread_id':'colab-d4'},'recursion_limit':40}); print(r4.get('veredito')); print(r4.get('rascunho')); persist()

In [ ]:
# E-mail com retriever explicitamente conectado (o processar() original não faz isso)
dia('dia2'); import indexar as ie; be=ie.construir(False)
dia('dia4'); import email_assistente as ea; from langgraph.types import Command
ge=ea.construir_grafo(retriever=be); emails=json.loads((N/'dados/emails.json').read_text()); cfg={'configurable':{'thread_id':'email-0'}}; st=ge.invoke(emails[0],cfg); snap=ge.get_state(cfg)
if snap.tasks and snap.tasks[0].interrupts: st=ge.invoke(Command(resume={'acao':'descartar'}),cfg)
print(st); persist()

# Dia 5 — Hugging Face + avaliação
**Falha atual do código original:** `avaliacao.py` importa `responder` de `dia5/cli.py`, mas essa função não existe. Além disso, `cli.py` continua usando a equipe do Dia 4 e não integra `modelos.py`, `embeddings_hf.py` ou `roteador.py`. O código HF original também fixa/assume CPU em vários pontos. As células abaixo tornam CUDA explícita sem esconder essas lacunas.

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('CUDA indisponível')
print(torch.cuda.get_device_name(0)); dia('dia5'); import modelos as m5; print({p:m5.para(p).model for p in ('supervisor','pesquisador','analista','redator','critico')})
from langchain_huggingface import HuggingFaceEmbeddings
emb=HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-small',model_kwargs={'device':'cuda'},encode_kwargs={'normalize_embeddings':True}); print('dim',len(emb.embed_query('faturamento 2024')))

In [ ]:
from transformers import pipeline
dia('dia5'); import pipelines as p5
sent=pipeline('sentiment-analysis',model=p5.MODELO_SENTIMENTO,device=0); print(sent(['Excelente equipamento','Atendimento péssimo'],truncation=True))
z=pipeline('zero-shot-classification',model=p5.MODELO_ZEROSHOT,device=0); print(z('Preciso comprar um torno industrial',candidate_labels=p5.CATEGORIAS))

In [ ]:
# Adapter para o contrato exigido por dia5/avaliacao.py
dia('dia4'); import equipe as ev; from langchain_core.messages import HumanMessage
gav=ev.compilar(str(N/'equipe_avaliacao_colab.db'))
def responder(pergunta):
    e={'messages':[HumanMessage(pergunta)],'pergunta':pergunta,'proximo':'','instrucao':pergunta,'achados':[],'rascunho':'','veredito':'','rodadas':0}
    r=gav.invoke(e,{'configurable':{'thread_id':f'eval-{uuid.uuid4()}'},'recursion_limit':40}); txt=r.get('rascunho') or r['messages'][-1].content or ''; fontes=sorted(set(re.findall(r'\[fonte:\s*([^\]]+)\]',txt,re.I))); return txt,fontes,0
dia('dia5'); import avaliacao as av; casos=av.carregar_casos(N/'avaliacao/casos.jsonl'); print('casos',len(casos))
RODAR_10=False
if RODAR_10:
    linhas=av.rodar(responder,casos); print(av.tabela(linhas)); (N/'saida').mkdir(exist_ok=True); (N/'saida/avaliacao_colab.json').write_text(json.dumps({'resumo':av.resumir(linhas),'linhas':linhas},ensure_ascii=False,indent=2))
persist()

## Finalização
Execute a última célula antes de encerrar o runtime. Para considerar Dias 2–5 validados, registre GPU, `ollama ps`, `pip freeze`, commit Git, uma execução de sucesso/erro por dia e os 10 casos de avaliação.

In [ ]:
persist(); print('Drive:',D); run(['du','-sh',str(D)],False); print(run(['ollama','ps'],False,True,env=os.environ.copy()).stdout)